# 01 — Join Strategies Deep Dive

PySpark join strategy notebook for Spark 4.x.

Focus:
- join strategy selection
- broadcast hash join
- sort-merge join
- shuffled hash join
- nested-loop joins
- join hints
- shuffle partition impact
- reading physical plans


## 00 — Setup

```text
Notebook
  |
  +-- SparkSession
        |
        +-- warehouse: /tmp/spark_join_strategies_warehouse
        +-- shuffle partitions: controlled per step
        +-- AQE: disabled first for deterministic plans
```


In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import types as T
from pathlib import Path
import shutil
import time

try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

warehouse_dir = "/tmp/spark_join_strategies_warehouse"
shutil.rmtree(warehouse_dir, ignore_errors=True)
Path(warehouse_dir).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("join-strategies-deep-dive")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

spark.version, spark.conf.get("spark.sql.warehouse.dir")


('4.0.2', 'file:/tmp/spark_join_strategies_warehouse')

## 01 — Sample warehouse-style data

```text
             customers
          +-------------+
          | customer_id |
          | segment     |
          | country     |
          +------+------+        fact_sales
                 |              +-------------+
                 +------------->| customer_id |
                                | product_id  |
             products           | store_id    |
          +-------------+       | quantity    |
          | product_id  |<------+ unit_price  |
          | category    |       +-------------+
          +-------------+
```


In [2]:
customers = spark.createDataFrame(
    [
        (1, "Enterprise", "US"),
        (2, "SMB", "DE"),
        (3, "Consumer", "SK"),
        (4, "Enterprise", "UK"),
        (5, "SMB", "CZ"),
        (6, "Consumer", "FR"),
    ],
    ["customer_id", "segment", "country"]
)

products = spark.createDataFrame(
    [
        (10, "Laptop", "Hardware"),
        (11, "Monitor", "Hardware"),
        (12, "Keyboard", "Accessories"),
        (13, "Mouse", "Accessories"),
        (14, "Support", "Service"),
    ],
    ["product_id", "product_name", "category"]
)

stores = spark.createDataFrame(
    [
        (100, "Online", "Global"),
        (101, "Bratislava", "SK"),
        (102, "Berlin", "DE"),
    ],
    ["store_id", "store_name", "store_country"]
)

fact_sales = (
    spark.range(1, 200001)
    .withColumnRenamed("id", "sale_id")
    .withColumn("customer_id", (F.col("sale_id") % 6 + 1).cast("int"))
    .withColumn("product_id", (F.col("sale_id") % 5 + 10).cast("int"))
    .withColumn("store_id", (F.col("sale_id") % 3 + 100).cast("int"))
    .withColumn("quantity", (F.col("sale_id") % 4 + 1).cast("int"))
    .withColumn("unit_price", (F.col("sale_id") % 900 + 100).cast("double"))
)

customers.cache().count()
products.cache().count()
stores.cache().count()
fact_sales.cache().count()

fact_sales.show(5)


+-------+-----------+----------+--------+--------+----------+
|sale_id|customer_id|product_id|store_id|quantity|unit_price|
+-------+-----------+----------+--------+--------+----------+
|      1|          2|        11|     101|       2|     101.0|
|      2|          3|        12|     102|       3|     102.0|
|      3|          4|        13|     100|       4|     103.0|
|      4|          5|        14|     101|       1|     104.0|
|      5|          6|        10|     102|       2|     105.0|
+-------+-----------+----------+--------+--------+----------+
only showing top 5 rows


## 02 — Baseline inner join

```text
fact_sales                         customers
+-------------+                    +-------------+
| customer_id |  equality join     | customer_id |
+------+------+ -----------------> +------+------+
       |                                  |
       +------------ matched rows --------+
```


In [3]:
baseline_join = (
    fact_sales
    .join(customers, on="customer_id", how="inner")
    .select("sale_id", "customer_id", "segment", "country", "product_id", "quantity")
)

baseline_join.show(5)
baseline_join.explain(mode="formatted")


+-------+-----------+----------+-------+----------+--------+
|sale_id|customer_id|   segment|country|product_id|quantity|
+-------+-----------+----------+-------+----------+--------+
|      1|          2|       SMB|     DE|        11|       2|
|      2|          3|  Consumer|     SK|        12|       3|
|      3|          4|Enterprise|     UK|        13|       4|
|      4|          5|       SMB|     CZ|        14|       1|
|      5|          6|  Consumer|     FR|        10|       2|
+-------+-----------+----------+-------+----------+--------+
only showing top 5 rows
== Physical Plan ==
* Project (12)
+- * BroadcastHashJoin Inner BuildRight (11)
   :- * Filter (5)
   :  +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
   :        +- InMemoryRelation (2)
   :              +- * Project (4)
   :                 +- * Range (3)
   +- BroadcastExchange (10)
      +- * Filter (9)
         +- InMemoryTableScan (6)
               +- InMemoryRelation (7)
                     +- * Sca

## 03 — Automatic BroadcastHashJoin

```text
Small dimension table
customers
+-------------+
| 6 rows      |
+------+------+       broadcast to executors
       |              +-------------------+
       +------------> | Executor 1 memory |
       +------------> | Executor 2 memory |
       +------------> | Executor N memory |

Large fact table stays partitioned.
Each partition probes local broadcast hash table.
```


In [4]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
spark.conf.set("spark.sql.adaptive.enabled", "false")

auto_broadcast_join = fact_sales.join(customers, on="customer_id", how="inner")

auto_broadcast_join.explain(mode="formatted")


== Physical Plan ==
* Project (12)
+- * BroadcastHashJoin Inner BuildRight (11)
   :- * Filter (5)
   :  +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
   :        +- InMemoryRelation (2)
   :              +- * Project (4)
   :                 +- * Range (3)
   +- BroadcastExchange (10)
      +- * Filter (9)
         +- InMemoryTableScan (6)
               +- InMemoryRelation (7)
                     +- * Scan ExistingRDD (8)


(1) InMemoryTableScan
Output [6]: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15]
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], [isnotnull(customer_id#11)]

(2) InMemoryRelation
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], StorageLevel(disk, memory, deserialized, 1 replicas), [sale_id#10L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#9L]
Arguments: Range (1, 200001, step=1, splits=Some(2))

(

## 04 — Explicit broadcast hint

```text
fact_sales partitions
 P0  P1  P2  P3
 |   |   |   |
 v   v   v   v
 local lookup against broadcast(customers)
```


In [5]:
hinted_broadcast_join = fact_sales.join(
    F.broadcast(customers),
    on="customer_id",
    how="inner"
)

hinted_broadcast_join.explain(mode="formatted")


== Physical Plan ==
* Project (12)
+- * BroadcastHashJoin Inner BuildRight (11)
   :- * Filter (5)
   :  +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
   :        +- InMemoryRelation (2)
   :              +- * Project (4)
   :                 +- * Range (3)
   +- BroadcastExchange (10)
      +- * Filter (9)
         +- InMemoryTableScan (6)
               +- InMemoryRelation (7)
                     +- * Scan ExistingRDD (8)


(1) InMemoryTableScan
Output [6]: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15]
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], [isnotnull(customer_id#11)]

(2) InMemoryRelation
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], StorageLevel(disk, memory, deserialized, 1 replicas), [sale_id#10L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#9L]
Arguments: Range (1, 200001, step=1, splits=Some(2))

(

## 05 — Force SortMergeJoin by disabling broadcast

```text
fact_sales                         customers
    |                                  |
    v                                  v
shuffle by customer_id           shuffle by customer_id
    |                                  |
    v                                  v
sort within partition             sort within partition
    |                                  |
    +------------ merge ---------------+
```


In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

sort_merge_join = fact_sales.join(customers, on="customer_id", how="inner")

sort_merge_join.explain(mode="formatted")


== Physical Plan ==
* Project (15)
+- * SortMergeJoin Inner (14)
   :- * Sort (7)
   :  +- Exchange (6)
   :     +- * Filter (5)
   :        +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
   :              +- InMemoryRelation (2)
   :                    +- * Project (4)
   :                       +- * Range (3)
   +- * Sort (13)
      +- Exchange (12)
         +- * Filter (11)
            +- InMemoryTableScan (8)
                  +- InMemoryRelation (9)
                        +- * Scan ExistingRDD (10)


(1) InMemoryTableScan
Output [6]: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15]
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], [isnotnull(customer_id#11)]

(2) InMemoryRelation
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], StorageLevel(disk, memory, deserialized, 1 replicas), [sale_id#10L ASC NULLS FIRST]

(3) Range [codegen id 

## 06 — ShuffledHashJoin hint

```text
fact_sales                         customers
    |                                  |
    v                                  v
shuffle by customer_id           shuffle by customer_id
    |                                  |
    v                                  v
probe large side                  build hash table per partition
    |                                  |
    +----------- hash join ------------+
```


In [7]:
shuffled_hash_join = fact_sales.hint("SHUFFLE_HASH").join(
    customers.hint("SHUFFLE_HASH"),
    on="customer_id",
    how="inner"
)

shuffled_hash_join.explain(mode="formatted")


== Physical Plan ==
* Project (13)
+- * ShuffledHashJoin Inner BuildRight (12)
   :- Exchange (6)
   :  +- * Filter (5)
   :     +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
   :           +- InMemoryRelation (2)
   :                 +- * Project (4)
   :                    +- * Range (3)
   +- Exchange (11)
      +- * Filter (10)
         +- InMemoryTableScan (7)
               +- InMemoryRelation (8)
                     +- * Scan ExistingRDD (9)


(1) InMemoryTableScan
Output [6]: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15]
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], [isnotnull(customer_id#11)]

(2) InMemoryRelation
Arguments: [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], StorageLevel(disk, memory, deserialized, 1 replicas), [sale_id#10L ASC NULLS FIRST]

(3) Range [codegen id : 1]
Output [1]: [id#9L]
Arguments: Range (1, 200001, s

## 07 — Shuffle partition impact

```text
Too few partitions
[ huge partition ][ huge partition ]

Balanced partitions
[ p0 ][ p1 ][ p2 ][ p3 ][ p4 ][ p5 ][ p6 ][ p7 ]

Too many partitions
[ tiny ][ tiny ][ tiny ][ tiny ][ tiny ][ tiny ][ tiny ][ tiny ][ ... ]
```


In [8]:
def timed_count(df, label):
    start = time.perf_counter()
    rows = df.count()
    elapsed = time.perf_counter() - start
    return (label, rows, round(elapsed, 3))

results = []

for partitions in [2, 8, 32]:
    spark.conf.set("spark.sql.shuffle.partitions", str(partitions))
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    df = fact_sales.join(customers, on="customer_id", how="inner")
    results.append(timed_count(df, f"shuffle_partitions={partitions}"))

spark.createDataFrame(results, ["scenario", "rows", "seconds"]).show(truncate=False)


+---------------------+------+-------+
|scenario             |rows  |seconds|
+---------------------+------+-------+
|shuffle_partitions=2 |200000|0.818  |
|shuffle_partitions=8 |200000|0.457  |
|shuffle_partitions=32|200000|0.644  |
+---------------------+------+-------+



## 08 — Multi-table join order

```text
              products
                 |
                 v
fact_sales -> join -> join -> join
                 ^       ^
                 |       |
             customers  stores

Small dimensions are good broadcast candidates.
```


In [9]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

star_join = (
    fact_sales
    .join(F.broadcast(customers), on="customer_id", how="inner")
    .join(F.broadcast(products), on="product_id", how="inner")
    .join(F.broadcast(stores), on="store_id", how="inner")
    .groupBy("segment", "category", "store_name")
    .agg(
        F.count("*").alias("sales_count"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("revenue")
    )
    .orderBy("segment", "category", "store_name")
)

star_join.show(20, truncate=False)
star_join.explain(mode="formatted")


+----------+-----------+----------+-----------+-----------+
|segment   |category   |store_name|sales_count|revenue    |
+----------+-----------+----------+-----------+-----------+
|Consumer  |Accessories|Berlin    |26667      |3.6576468E7|
|Consumer  |Hardware   |Berlin    |26667      |3.6576774E7|
|Consumer  |Service    |Berlin    |13333      |1.8588222E7|
|Enterprise|Accessories|Online    |26667      |3.6443738E7|
|Enterprise|Hardware   |Online    |26666      |3.6443638E7|
|Enterprise|Service    |Online    |13333      |1.8521258E7|
|SMB       |Accessories|Bratislava|26666      |3.6709794E7|
|SMB       |Hardware   |Bratislava|26667      |3.6709788E7|
|SMB       |Service    |Bratislava|13334      |1.825552E7 |
+----------+-----------+----------+-----------+-----------+

== Physical Plan ==
* Sort (31)
+- Exchange (30)
   +- * HashAggregate (29)
      +- Exchange (28)
         +- * HashAggregate (27)
            +- * Project (26)
               +- * BroadcastHashJoin Inner BuildRight (2

## 09 — Non-equi join and BroadcastNestedLoopJoin

```text
fact_sales.amount             discount_rules
     |                              |
     | amount between min and max   |
     +----------------------------->|

No simple equality key.
Broadcast nested-loop may be selected when one side is small.
```


In [10]:
discount_rules = spark.createDataFrame(
    [
        ("LOW", 0.0, 300.0, 0.00),
        ("MID", 300.0, 700.0, 0.05),
        ("HIGH", 700.0, 2000.0, 0.10),
    ],
    ["price_band", "min_amount", "max_amount", "discount_rate"]
)

sales_amounts = fact_sales.withColumn(
    "amount",
    F.col("quantity") * F.col("unit_price")
)

range_join = sales_amounts.join(
    F.broadcast(discount_rules),
    (F.col("amount") >= F.col("min_amount")) & (F.col("amount") < F.col("max_amount")),
    "inner"
)

range_join.select("sale_id", "amount", "price_band", "discount_rate").show(10)
range_join.explain(mode="formatted")


+-------+------+----------+-------------+
|sale_id|amount|price_band|discount_rate|
+-------+------+----------+-------------+
|      1| 202.0|       LOW|          0.0|
|      2| 306.0|       MID|         0.05|
|      3| 412.0|       MID|         0.05|
|      4| 104.0|       LOW|          0.0|
|      5| 210.0|       LOW|          0.0|
|      6| 318.0|       MID|         0.05|
|      7| 428.0|       MID|         0.05|
|      8| 108.0|       LOW|          0.0|
|      9| 218.0|       LOW|          0.0|
|     10| 330.0|       MID|         0.05|
+-------+------+----------+-------------+
only showing top 10 rows
== Physical Plan ==
* BroadcastNestedLoopJoin Inner BuildRight (10)
:- * Project (6)
:  +- * Filter (5)
:     +- InMemoryTableScan (1) (columnarIn=false, columnarOut=true)
:           +- InMemoryRelation (2)
:                 +- * Project (4)
:                    +- * Range (3)
+- BroadcastExchange (9)
   +- * Filter (8)
      +- * Scan ExistingRDD (7)


(1) InMemoryTableScan
Output [

## 10 — Hint comparison

```text
Same logical join
       |
       +-- BROADCAST hint       -> BroadcastHashJoin if applicable
       +-- MERGE hint           -> SortMergeJoin
       +-- SHUFFLE_HASH hint    -> ShuffledHashJoin if applicable
       +-- SHUFFLE_REPLICATE_NL -> Cartesian-like replicated nested-loop pattern
```


In [12]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.shuffle.partitions", "8")

hint_cases = {
    "broadcast_hint": fact_sales.join(F.broadcast(customers), "customer_id"),
    "merge_hint": fact_sales.hint("MERGE").join(customers.hint("MERGE"), "customer_id"),
    "shuffle_hash_hint": fact_sales.hint("SHUFFLE_HASH").join(customers.hint("SHUFFLE_HASH"), "customer_id"),
}

for name, df in hint_cases.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    df.explain(mode="simple")



broadcast_hint
== Physical Plan ==
*(2) Project [customer_id#11, sale_id#10L, product_id#12, store_id#13, quantity#14, unit_price#15, segment#1, country#2]
+- *(2) BroadcastHashJoin [cast(customer_id#11 as bigint)], [customer_id#0L], Inner, BuildRight, false
   :- *(2) Filter isnotnull(customer_id#11)
   :  +- InMemoryTableScan [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], [isnotnull(customer_id#11)]
   :        +- InMemoryRelation [sale_id#10L, customer_id#11, product_id#12, store_id#13, quantity#14, unit_price#15], StorageLevel(disk, memory, deserialized, 1 replicas)
   :              +- *(1) Project [id#9L AS sale_id#10L, cast(((id#9L % 6) + 1) as int) AS customer_id#11, cast(((id#9L % 5) + 10) as int) AS product_id#12, cast(((id#9L % 3) + 100) as int) AS store_id#13, cast(((id#9L % 4) + 1) as int) AS quantity#14, cast(((id#9L % 900) + 100) as double) AS unit_price#15]
   :                 +- *(1) Range (1, 200001, step=1, splits=2)
   +- Br

## 11 — Strategy summary

```text
Decision shape

Is one side small enough?
        |
        +-- yes --> BroadcastHashJoin
        |
        +-- no --> Is equality join?
                    |
                    +-- yes --> SortMergeJoin / ShuffledHashJoin
                    |
                    +-- no  --> BroadcastNestedLoopJoin / Cartesian-like strategy
```


In [13]:
summary = spark.createDataFrame(
    [
        ("BroadcastHashJoin", "One side is small", "Avoids shuffle on large side", "Can fail or spill if broadcast side is too large"),
        ("SortMergeJoin", "Large equality joins", "Stable default for big data", "Requires shuffle and sort"),
        ("ShuffledHashJoin", "Equality joins with build side per partition", "Can avoid sort", "More sensitive to memory pressure"),
        ("BroadcastNestedLoopJoin", "Non-equi joins with small side", "Works without equality key", "Can be expensive for large inputs"),
        ("CartesianProduct", "Cross joins", "Explicit all-pairs semantics", "Usually dangerous at scale"),
    ],
    ["strategy", "best_for", "strength", "risk"]
)

summary.show(truncate=False)


+-----------------------+--------------------------------------------+----------------------------+------------------------------------------------+
|strategy               |best_for                                    |strength                    |risk                                            |
+-----------------------+--------------------------------------------+----------------------------+------------------------------------------------+
|BroadcastHashJoin      |One side is small                           |Avoids shuffle on large side|Can fail or spill if broadcast side is too large|
|SortMergeJoin          |Large equality joins                        |Stable default for big data |Requires shuffle and sort                       |
|ShuffledHashJoin       |Equality joins with build side per partition|Can avoid sort              |More sensitive to memory pressure               |
|BroadcastNestedLoopJoin|Non-equi joins with small side              |Works without equality key  |Can be 